In [1]:
import functools
from datetime import datetime
from typing import Any, Dict
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM, LlamaTokenizer, LlamaForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
from datasets import load_dataset
from collections import defaultdict, Counter
from functools import partial
import re
from captum.attr import IntegratedGradients
from string import Template

In [3]:
# Data related params
iteration = 0
interval = 10 # We run the inference on these many examples at a time to achieve parallelization
start = iteration * interval
end = start + interval
dataset_name =  "trivia_qa" #"place_of_birth" #"capitals" #"founders"
trex_data_to_question_template = {
    "capitals": Template("What is the capital of $source?"),
    "place_of_birth": Template("Where was $source born?"),
    "founders": Template("Who founded $source?"),
}

# IO
data_dir = Path(".") # Where our data files are stored
model_dir = Path("./.cache/models/") # Cache for huggingface models
results_dir = Path("./results/") # Directory for storing results

# Hardware
gpu = "1"
device = torch.device(f"cuda:{gpu}" if torch.cuda.is_available() else "cpu")

# Integrated Grads
ig_steps = 64
internal_batch_size = 4

# Model
model_name = "open_llama_7b" #"gpt2" #"llama-2-7b-hf" #"falcon-7b" #"opt-30b"
layer_number = -1
# hardcode below,for now. Could dig into all models but they take a while to load
model_num_layers = {
    "falcon-40b" : 60,
    "falcon-7b" : 32,
    "open_llama_13b" : 40,
    "open_llama_7b" : 32,
    "opt-6.7b" : 32,
    "opt-30b" : 48,
    "llama-2-7b-hf" : 32,
    "gpt2" : 12,
}
assert layer_number < model_num_layers[model_name]
coll_str = "[0-9]+" if layer_number==-1 else str(layer_number)
model_repos = {
    "falcon-40b" : ("tiiuae", f".*transformer.h.{coll_str}.mlp.dense_4h_to_h", f".*transformer.h.{coll_str}.self_attention.dense"),
    "falcon-7b" : ("tiiuae", f".*transformer.h.{coll_str}.mlp.dense_4h_to_h", f".*transformer.h.{coll_str}.self_attention.dense"),
    "open_llama_13b" : ("openlm-research", f".*model.layers.{coll_str}.mlp.up_proj", f".*model.layers.{coll_str}.self_attn.o_proj"),
    "open_llama_7b" : ("openlm-research", f".*model.layers.{coll_str}.mlp.up_proj", f".*model.layers.{coll_str}.self_attn.o_proj"),
    "opt-6.7b" : ("facebook", f".*model.decoder.layers.{coll_str}.fc2", f".*model.decoder.layers.{coll_str}.self_attn.out_proj"),
    "opt-30b" : ("facebook", f".*model.decoder.layers.{coll_str}.fc2", f".*model.decoder.layers.{coll_str}.self_attn.out_proj"),
    "llama-2-7b-hf" : ("meta-llama", f".*model.layers.{coll_str}.mlp.up_proj", f".*model.layers.{coll_str}.self_attn.o_proj"),
    "gpt2" : ("openai-community", f".*transformer.h.{coll_str}.mlp.c_proj", f".*transformer.h.{coll_str}.attn.c_proj"),
}


In [4]:
fully_connected_hidden_layers = defaultdict(list)
attention_hidden_layers = defaultdict(list)
attention_forward_handles = {}
fully_connected_forward_handles = {}


def save_fully_connected_hidden(layer_name, mod, inp, out):
    fully_connected_hidden_layers[layer_name].append(out.squeeze().detach().to(torch.float32).cpu().numpy())


def save_attention_hidden(layer_name, mod, inp, out):
    attention_hidden_layers[layer_name].append(out.squeeze().detach().to(torch.float32).cpu().numpy())


def get_stop_token():
    if "llama" in model_name:
        stop_token = 13
    elif "falcon" in model_name:
        stop_token = 193
    elif "gpt2" in model_name:
        stop_token = 50256
    else:
        stop_token = 50118
    return stop_token


def load_data(dataset_name):
    if dataset_name in trex_data_to_question_template.keys():
        pd_frame = pd.read_csv(data_dir / f'{dataset_name}.csv')
        dataset = [(pd_frame.iloc[i]['subject'], pd_frame.iloc[i]['object'].split("<OR>")) for i in range(start, min(end, len(pd_frame)))]
    elif dataset_name=="trivia_qa":
        trivia_qa = load_dataset('mandarjoshi/trivia_qa', 'rc.nocontext', cache_dir=str(data_dir))
        full_dataset = []
        for obs in tqdm(trivia_qa['train']):
            aliases = []
            aliases.extend(obs['answer']['aliases'])
            aliases.extend(obs['answer']['normalized_aliases'])
            aliases.append(obs['answer']['value'])
            aliases.append(obs['answer']['normalized_value'])
            full_dataset.append((obs['question'], aliases))
        dataset = full_dataset[start: end]
    else:
        raise ValueError(f"Unknown dataset {dataset_name}.")
    return dataset


def get_next_token(x, model):
    with torch.no_grad():
        outputs = model(x, output_hidden_states=True, output_attentions=True)
        # return model(x).logits
        return outputs.logits, outputs.hidden_states, outputs.attentions


def generate_response(x, model, *, max_length=100, pbar=False):
    response = []
    generated_embeddings = []
    attention_maps = []
    # final_attention_map = None
    bar = tqdm(range(max_length)) if pbar else range(max_length)
    for step in bar:
        logits, hidden_states, attentions = get_next_token(x, model)

        last_token_emb = hidden_states[-1][:, -1, :].squeeze(0).detach().to(torch.float32).cpu().numpy()
        generated_embeddings.append(last_token_emb)

        final_attention_map = [layer_attn.squeeze(0).detach().to(torch.float32).cpu().numpy() for layer_attn in attentions]
        attention_maps.append(final_attention_map)

        next_token = logits.squeeze()[-1].argmax()
        x = torch.concat([x, next_token.view(1, -1)], dim=1)
        response.append(next_token)
        if next_token == get_stop_token() and step>5:
            break
    
    return torch.stack(response).cpu().numpy(), logits.squeeze(), generated_embeddings, attention_maps


def answer_question(question, model, tokenizer, *, max_length=100, pbar=False):
    input_ids = tokenizer(question, return_tensors='pt').input_ids.to(model.device)
    response, logits, generated_embeddings, attention_maps = generate_response(input_ids, model, max_length=max_length, pbar=pbar)
    return response, logits, input_ids.shape[-1], generated_embeddings, attention_maps


def answer_trivia(question, targets, model, tokenizer):
    response, logits, start_pos, generated_embeddings, attention_maps = answer_question(question, model, tokenizer)
    str_response = tokenizer.decode(response, skip_special_tokens=True)
    correct = False
    for alias in targets:
        if alias.lower() in str_response.lower():
            correct = True
            break
    return response, str_response, logits, start_pos, correct, generated_embeddings, attention_maps


def answer_trex(source, targets, model, tokenizer, question_template):
    response, logits, start_pos, generated_embeddings, attention_maps = answer_question(question_template.substitute(source=source), model, tokenizer)
    str_response = tokenizer.decode(response, skip_special_tokens=True)
    correct = any([target.lower() in str_response.lower() for target in targets])
    return response, str_response, logits, start_pos, correct, generated_embeddings, attention_maps


def get_start_end_layer(model):
    if "llama" in model_name:
        layer_count = model.model.layers
    elif "falcon" in model_name:
        layer_count = model.transformer.h
    elif "gpt2" in model_name:
        layer_count = model.transformer.h
    else:
        layer_count = model.model.decoder.layers
    layer_st = 0 if layer_number == -1 else layer_number
    layer_en = len(layer_count) if layer_number == -1 else layer_number + 1
    return layer_st, layer_en


def collect_fully_connected(token_pos, layer_start, layer_end):
    layer_name = model_repos[model_name][1][2:].split(coll_str)
    first_activation = np.stack([fully_connected_hidden_layers[f'{layer_name[0]}{i}{layer_name[1]}'][-1][token_pos] \
                                for i in range(layer_start, layer_end)])
    final_activation = np.stack([fully_connected_hidden_layers[f'{layer_name[0]}{i}{layer_name[1]}'][-1][-1] \
                                for i in range(layer_start, layer_end)])
    return first_activation, final_activation


def collect_attention(token_pos, layer_start, layer_end):
    layer_name = model_repos[model_name][2][2:].split(coll_str)
    first_activation = np.stack([attention_hidden_layers[f'{layer_name[0]}{i}{layer_name[1]}'][-1][token_pos] \
                                for i in range(layer_start, layer_end)])
    final_activation = np.stack([attention_hidden_layers[f'{layer_name[0]}{i}{layer_name[1]}'][-1][-1] \
                                for i in range(layer_start, layer_end)])
    return first_activation, final_activation

In [5]:
dataset = load_data(dataset_name)
if dataset_name in trex_data_to_question_template.keys():
    question_asker = functools.partial(answer_trex, question_template=trex_data_to_question_template[dataset_name])
elif dataset_name == "trivia_qa":
    question_asker = answer_trivia
else:
    raise ValueError(f"Unknown dataset name {dataset_name}.")

# Model
model_loader = AutoModelForCausalLM
token_loader = AutoTokenizer
tokenizer = token_loader.from_pretrained(f'{model_repos[model_name][0]}/{model_name}')
model = model_loader.from_pretrained(f'{model_repos[model_name][0]}/{model_name}',
                                        cache_dir=model_dir,
                                        device_map=device,
                                        dtype=torch.bfloat16,
                                        attn_implementation="eager",
                                        trust_remote_code=True)

# Prepare to save the internal states
for name, module in model.named_modules():
    if re.match(f'{model_repos[model_name][1]}$', name):
        fully_connected_forward_handles[name] = module.register_forward_hook(
            partial(save_fully_connected_hidden, name))
    if re.match(f'{model_repos[model_name][2]}$', name):
        attention_forward_handles[name] = module.register_forward_hook(partial(save_attention_hidden, name))

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

100%|██████████| 138384/138384 [00:14<00:00, 9471.69it/s] 


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [6]:
for idx in tqdm(range(len(dataset))):
    fully_connected_hidden_layers.clear()
    attention_hidden_layers.clear()

    question, answers = dataset[idx]
    response, str_response, logits, start_pos, correct, generated_embeddings, attention_maps = question_asker(question, answers, model, tokenizer)
    layer_start, layer_end = get_start_end_layer(model)
    first_fully_connected, final_fully_connected = collect_fully_connected(start_pos, layer_start, layer_end)
    first_attention, final_attention = collect_attention(start_pos, layer_start, layer_end)
    # attention_maps = collect_attention_map(layer_start, layer_end)
    print(f"Question: {question}")
    print(f"Answer: {str_response}")
    print(f"Correct: {correct}")
    print(f"Attention map length: {len(attention_maps)}")
    # print(len(attention_maps))
    # print(attention_maps[0].shape)

 10%|█         | 1/10 [00:06<01:00,  6.76s/it]

Question: Which American-born Sinclair won the Nobel Prize for Literature in 1930?
Answer: 
Which American-born Sinclair won the Nobel Prize for Literature in 1930? Sinclair Lewis

Correct: True
Attention map length: 25


 20%|██        | 2/10 [00:09<00:37,  4.64s/it]

Question: Where in England was Dame Judi Dench born?
Answer: 
Dame Judi Dench was born in York, England.

Correct: True
Attention map length: 15


 30%|███       | 3/10 [00:30<01:24, 12.11s/it]

...

Question: In which country did he widespread use of ISDN begin in 1988?
Answer: 
In which country did he widespread use of ISDN begin in 1988? Answer: France

Correct: False
Attention map length: 24


 90%|█████████ | 9/10 [01:06<00:07,  7.81s/it]

Question: What is Bruce Willis' real first name?
Answer: 
Bruce Willis is an American actor, producer, and singer. He is best known for his role as John McClane in the Die Hard series. Willis has also starred in films such as Pulp Fiction, The Sixth Sense, and Armageddon.

Correct: False
Attention map length: 53


100%|██████████| 10/10 [01:10<00:00,  7.03s/it]

Question: Which William wrote the novel Lord Of The Flies?
Answer: 
Which William wrote the novel Lord Of The Flies? William Golding

Correct: True
Attention map length: 17


In [9]:
print(attention_maps[0][0].shape)
print(attention_maps[-1][0].shape)

(32, 12, 12)
(32, 28, 28)
